# Basline classifier using 1D CNN for time series data

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import torch
import torch.nn as nn
import copy
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from src.load import make_loader

In [2]:
PROCESSED_DIR = PROJECT_ROOT / 'dataset' / 'processed'
MODEL_DIR = PROJECT_ROOT / "models"
data = np.load(PROCESSED_DIR / 'splits.npz')

X_train = data['X_train']
X_val   = data['X_val']
X_test  = data['X_test']
y_train = data['y_train']
y_val   = data['y_val']
y_test  = data['y_test']

print('Train:', X_train.shape, 'labels:', y_train.shape)
print('Val  :', X_val.shape)
print('Test :', X_test.shape)

Train: (14525, 6, 300) labels: (14525,)
Val  : (2075, 6, 300)
Test : (4150, 6, 300)


In [3]:
train_loader = make_loader(X_train, y_train, batch_size=128, shuffle=True)
val_loader   = make_loader(X_val, y_val, batch_size=128)
test_loader  = make_loader(X_test, y_test, batch_size=128)

In [4]:
X, y = next(iter(train_loader))

print(f"X shape: {X.shape}\ny shape: {y.shape}")
print("y unique:", torch.unique(y))
print(f"X mean: {X.mean():.4f}\nX std: {X.std():.4f}")
print(f"X_train mean: {X_train.mean():.4f}\nX_train std: {X_train.std():.4f}")

X shape: torch.Size([128, 6, 300])
y shape: torch.Size([128])
y unique: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])
X mean: 0.0000
X std: 0.9999
X_train mean: -0.0000
X_train std: 0.9999


In [5]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=18):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(6, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.2),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(3),
            nn.Dropout(0.2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 25, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
    
def train_classifier(model, train_loader, val_loader, epochs=15, lr=1e-3, device="cpu"):
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    best_acc = 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for X, y in train_loader:
            X, y = X.to(device), y.to(device)

            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        model.eval()
        correct, total = 0, 0

        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                out = model(X)
                preds = out.argmax(dim=1)

                correct += (preds == y).sum().item()
                total += y.size(0)

        acc = correct / total
        avg_loss = total_loss / len(train_loader)
        
        print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Val Acc: {acc:.4f}")
        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model

In [9]:
device="cpu"
model = BaselineCNN()
model = train_classifier(
    model,
    train_loader,
    val_loader,
    epochs=15,
    lr=1e-3,
    device=device
)
torch.save(model.state_dict(), MODEL_DIR / 'baseline_classifier.pth')

Epoch 1 | Loss: 1.7182 | Val Acc: 0.5865
Epoch 2 | Loss: 1.0323 | Val Acc: 0.5947
Epoch 3 | Loss: 0.8290 | Val Acc: 0.6814
Epoch 4 | Loss: 0.7074 | Val Acc: 0.7533
Epoch 5 | Loss: 0.6247 | Val Acc: 0.7990
Epoch 6 | Loss: 0.5764 | Val Acc: 0.7875
Epoch 7 | Loss: 0.5276 | Val Acc: 0.7687
Epoch 8 | Loss: 0.4915 | Val Acc: 0.8159
Epoch 9 | Loss: 0.4541 | Val Acc: 0.7942
Epoch 10 | Loss: 0.4366 | Val Acc: 0.8159
Epoch 11 | Loss: 0.4212 | Val Acc: 0.8275
Epoch 12 | Loss: 0.3948 | Val Acc: 0.8699
Epoch 13 | Loss: 0.3815 | Val Acc: 0.8655
Epoch 14 | Loss: 0.3641 | Val Acc: 0.8713
Epoch 15 | Loss: 0.3452 | Val Acc: 0.8834


## Evaluation

In [10]:
def evaluate(model, loader, device="cpu"):
    model.eval()

    all_preds, all_labels = [], []

    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            out = model(X)
            preds = out.argmax(dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(y.numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro')

    return acc, f1

In [11]:
model = BaselineCNN().to(device="cpu")
model.load_state_dict(torch.load(MODEL_DIR / "baseline_classifier.pth"))
model.eval()

test_acc, test_f1 = evaluate(model, test_loader)
print("Test Accuracy:", test_acc)
print("Test Macro F1:", test_f1)

Test Accuracy: 0.8795180722891566
Test Macro F1: 0.8757469843434043


In [12]:
def full_report(model, loader):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            out = model(X)
            preds = out.argmax(dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(y.numpy())

    print(classification_report(all_labels, all_preds))

full_report(model, test_loader)

              precision    recall  f1-score   support

           0       0.76      0.69      0.72       377
           1       0.73      0.77      0.75       375
           2       0.83      0.95      0.89       359
           3       0.88      0.97      0.92       373
           4       0.97      0.99      0.98       436
           5       0.88      0.79      0.83       363
           6       0.89      0.96      0.92       352
           7       0.98      0.88      0.93       267
           8       0.98      0.98      0.98       133
           9       1.00      0.81      0.90        96
          10       0.87      0.99      0.93       201
          11       0.91      0.93      0.92       176
          12       0.97      0.92      0.94        63
          13       0.79      0.44      0.57        52
          14       0.99      0.87      0.93       119
          15       0.87      0.92      0.89       160
          16       0.94      0.80      0.87       156
          17       0.95    

## 20% labeled data

In [13]:
def sample_labels(X, y, fraction, random_state=42):
    if fraction >= 1.0:
        return X, y
    _, X_f, _, y_f = train_test_split(X, y, test_size=fraction, stratify=y, random_state=random_state)
    return X_f, y_f

In [14]:
X_small, y_small = sample_labels(X_train, y_train, fraction=0.2)
train_loader_small = make_loader(X_small, y_small, batch_size=128, shuffle=True)

classifier_small = BaselineCNN()
model_small = train_classifier(
    classifier_small,
    train_loader_small,
    val_loader,
    epochs=15,
    lr=1e-3,
    device=device
)

torch.save(model_small.state_dict(), MODEL_DIR / 'baseline_20.pth')

Epoch 1 | Loss: 2.2829 | Val Acc: 0.3104
Epoch 2 | Loss: 1.8200 | Val Acc: 0.4183
Epoch 3 | Loss: 1.5270 | Val Acc: 0.4607
Epoch 4 | Loss: 1.2988 | Val Acc: 0.5186
Epoch 5 | Loss: 1.1005 | Val Acc: 0.5793
Epoch 6 | Loss: 0.9981 | Val Acc: 0.5889
Epoch 7 | Loss: 0.8879 | Val Acc: 0.6116
Epoch 8 | Loss: 0.7793 | Val Acc: 0.6347
Epoch 9 | Loss: 0.7498 | Val Acc: 0.6612
Epoch 10 | Loss: 0.6771 | Val Acc: 0.6414
Epoch 11 | Loss: 0.6421 | Val Acc: 0.6993
Epoch 12 | Loss: 0.6094 | Val Acc: 0.6689
Epoch 13 | Loss: 0.5835 | Val Acc: 0.7113
Epoch 14 | Loss: 0.5344 | Val Acc: 0.6501
Epoch 15 | Loss: 0.5345 | Val Acc: 0.6973


In [15]:
# Evaluate
model_small = BaselineCNN().to(device)
model_small.load_state_dict(torch.load(MODEL_DIR / "baseline_20.pth"))

test_acc, test_f1 = evaluate(model_small, test_loader)

print("Baseline 20% -> Test Acc:", test_acc)
print("Baseline 20% -> Test F1 :", test_f1)

Baseline 20% -> Test Acc: 0.7187951807228916
Baseline 20% -> Test F1 : 0.6856706555368647
